# DNN Deep SVDD — Train from Scratch

Pipeline: Generate synthetic tabular data → Initialize center **c** →
Deep SVDD train (minimise ‖f(x) − c‖²) → Compute R² → Save weights.

All code is self-contained — no dependency on the `padi` package.

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

REPO = Path.cwd()
while REPO.name and not (REPO / "pythonsi").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from network import MLP

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# --- CONFIGURATION ---
SEED = 42
N_FEATURES = 20
N_TRAIN = 1000
HIDDEN_DIM = 32
REP_DIM = 8
BATCH_SIZE = 64
SVDD_EPOCHS = 100
NU = 0.05

torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Generate Synthetic Training Data

In [ ]:
# Normal tabular data: N(0, 1) in each feature
X_train = np.random.normal(size=(N_TRAIN, N_FEATURES)).astype(np.float32)
train_tensor = torch.from_numpy(X_train)
print(f"Training data: {train_tensor.shape}")

## 2. Initialize Center **c** & Train Deep SVDD

In [ ]:
net = MLP(
    n_features=N_FEATURES,
    hidden_dim=HIDDEN_DIM,
    repdim=REP_DIM,
).to(DEVICE)

train_dataset = TensorDataset(train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# --- Initialize center c = mean of encoder outputs ---
net.eval()
all_outputs = []
with torch.no_grad():
    for (batch,) in DataLoader(train_dataset, batch_size=BATCH_SIZE):
        all_outputs.append(net(batch.to(DEVICE)).cpu())
all_outputs = torch.cat(all_outputs, dim=0)
center_c = all_outputs.mean(dim=0).to(DEVICE)
print(f"Center c shape: {center_c.shape}")

# --- Deep SVDD training ---
net.train()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-6)

print("\nDeep SVDD training …")
for epoch in range(SVDD_EPOCHS):
    total_loss = 0.0
    for (batch,) in train_loader:
        batch = batch.to(DEVICE)
        outputs = net(batch)
        dist = torch.sum((outputs - center_c) ** 2, dim=1)
        loss = torch.mean(dist)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(
            f"  Epoch {epoch + 1}/{SVDD_EPOCHS} | Loss: {total_loss / len(train_loader):.8f}"
        )

print("Deep SVDD training done.")

## 3. Compute R² and Save Weights

In [ ]:
net.eval()
all_dists = []
with torch.no_grad():
    for (batch,) in DataLoader(train_dataset, batch_size=BATCH_SIZE):
        outputs = net(batch.to(DEVICE))
        dist = torch.sum((outputs - center_c) ** 2, dim=1)
        all_dists.append(dist.cpu().numpy())

all_dists = np.concatenate(all_dists)
R_squared = float(np.quantile(all_dists, 1 - NU))
print(f"R² = {R_squared:.10f}")
print(f"Center c: {center_c.cpu().numpy()}")

# Save
os.makedirs("weights", exist_ok=True)
torch.save(
    {
        "model_state_dict": net.state_dict(),
        "center_c": center_c.cpu().numpy(),
        "R_squared": R_squared,
        "config": {
            "n_features": N_FEATURES,
            "hidden_dim": HIDDEN_DIM,
            "repdim": REP_DIM,
        },
    },
    "weights/mlp_encoder.pth",
)
print("Weights saved to weights/mlp_encoder.pth")